# Mammography — post hoc transfer learning

Exploratory evaluation of ImageNet-initialized Swin-Tiny under the established mammography ROI protocol.


In [ ]:
from pathlib import Path
import numpy as np

# Leave this path as None for automatic discovery under /kaggle/input.
# Accepts either ROI_Crops_256_v1_Kaggle.zip or an extracted folder containing roi_crop_manifest.csv.
SOURCE_PATH = None

OUT_DIR = Path("/kaggle/working/ROI256_WAVE2BIS_TRANSFER_RESULTS")

# Recommended Kaggle input filename: swin_t-704ceda3.pth
# PRETRAINED_WEIGHTS_PATH = "/kaggle/input/swin-t-imagenet/swin_t-704ceda3.pth"
PRETRAINED_WEIGHTS_PATH = None
AUTO_FIND_PRETRAINED_WEIGHTS = True
ALLOW_INTERNET_DOWNLOAD = False  # Keep False for offline, reproducible Kaggle execution.

RUN_TRAINING = True
RUN_VISUALIZATION = True
RUN_FAST_DEV = False

# Keep the scratch baseline disabled unless a repeat run is required.
# Enable only when repeating the scratch baseline in the same notebook.
RUN_SCRATCH_BASELINE_RERUN = False

# CLAHE remained disabled by default after the Wave 2 result; enable only for the optional ablation.
RUN_OPTIONAL_CLAHE_ARM = False

CONFIGS_TO_RUN = []
if RUN_SCRATCH_BASELINE_RERUN:
    CONFIGS_TO_RUN.append("scratch_baseline_rerun")
CONFIGS_TO_RUN += [
    "pretrained_light",
    "pretrained_augmod",
    "pretrained_augmod_swa",
]
if RUN_OPTIONAL_CLAHE_ARM:
    CONFIGS_TO_RUN.append("pretrained_augmod_clahe")

TRAINING_SEEDS = [42, 123, 2025]

IMAGE_SIZE = 256
BATCH_SIZE = 8
NUM_WORKERS = 2
EPOCHS = 100
PATIENCE = 15
AMP = True

BASE_LR = 2e-4
ENCODER_LR = 2e-5
PRETRAINED_ENCODER_LR = 2e-5
WEIGHT_DECAY = 1e-4
WARMUP_EPOCHS = 5
GRAD_CLIP_NORM = 1.0

FREEZE_PRETRAINED_ENCODER_EPOCHS = 5

SWA_START_FRAC = 0.75
SWA_LR = 5e-5

# Select thresholds on CBIS validation data only.
THRESHOLDS = [round(float(x), 3) for x in np.arange(0.05, 0.96, 0.05)]

LIGHT_HFLIP_P = 0.5
LIGHT_ROT_DEG = 7.0
LIGHT_TRANSLATE_FRAC = 0.03
LIGHT_SCALE_LOW = 0.95
LIGHT_SCALE_HIGH = 1.05

MOD_HFLIP_P = 0.5
MOD_VFLIP_P = 0.25
MOD_ROT_DEG = 12
MOD_SHIFT_LIMIT = 0.06
MOD_SCALE_LIMIT = 0.08
MOD_BRIGHTNESS_CONTRAST_LIMIT = 0.12
MOD_GAMMA_LIMIT = (85, 115)

CLAHE_CLIP_LIMIT = 2.0
CLAHE_TILE_GRID_SIZE = (8, 8)

SAVE_QUALITATIVE_N = 16
PREPROCESSING_VIS_N = 6

# Baseline from the validated Track A result. Used only for comparison, not for selection.
BASELINE_REFERENCE = {
    "model": "swin_tiny_unet_scratch_previous",
    "cbis_test_dice": 0.888,
    "inbreast_dice": 0.849,
    "inbreast_hd95": 17.93,
}

print("Configuration loaded. Output directory:", OUT_DIR)
print("Configs to run:", CONFIGS_TO_RUN)
print("This notebook will not write to ROI256_TRAINING_RESULTS, ROI256_WAVE1_RESULTS, or ROI256_WAVE2_CLAHE_AUG_SWA_RESULTS.")

In [ ]:
import os, sys, json, math, time, random, shutil, zipfile, hashlib, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from IPython.display import display, Image as IPyImage

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn

try:
    import matplotlib.pyplot as plt
    MATPLOTLIB_AVAILABLE = True
except Exception as e:
    MATPLOTLIB_AVAILABLE = False
    print("matplotlib is unavailable:", repr(e))

try:
    from scipy.ndimage import binary_erosion, distance_transform_edt
    SCIPY_AVAILABLE = True
except Exception as e:
    SCIPY_AVAILABLE = False
    print("scipy is unavailable: HD95/ASD will be NaN", repr(e))

try:
    import torchvision
    import torchvision.transforms.functional as TF
    from torchvision.transforms import InterpolationMode
    TORCHVISION_AVAILABLE = True
except Exception as e:
    TORCHVISION_AVAILABLE = False
    print("torchvision is unavailable: Swin/augmentations are unavailable", repr(e))

try:
    import cv2
    CV2_AVAILABLE = True
except Exception as e:
    CV2_AVAILABLE = False
    print("OpenCV unavailable. Optional CLAHE arm will be disabled unless cv2 is installed.", repr(e))

try:
    import albumentations as A
    ALBUMENTATIONS_AVAILABLE = True
except Exception as e:
    ALBUMENTATIONS_AVAILABLE = False
    print("Albumentations unavailable. Moderate augmentation will fall back to torchvision affine/light transforms.", repr(e))

OUT_DIR.mkdir(parents=True, exist_ok=True)
for d in ["checkpoints", "metrics", "figures", "logs", "probabilities"]:
    (OUT_DIR / d).mkdir(exist_ok=True)

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Torchvision:", getattr(torchvision, "__version__", "NA") if TORCHVISION_AVAILABLE else "NA")
print("SciPy:", SCIPY_AVAILABLE)
print("OpenCV:", cv2.__version__ if CV2_AVAILABLE else "NA")
print("Albumentations:", A.__version__ if ALBUMENTATIONS_AVAILABLE else "NA")

In [ ]:
def seed_everything(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def worker_init_fn(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def sha256_file(path: Path, chunk_size=1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def now():
    return time.strftime("%Y-%m-%d %H:%M:%S")

In [ ]:
def find_manifest_in_folder(folder: Path):
    hits = list(Path(folder).rglob("roi_crop_manifest.csv"))
    if not hits:
        return None
    return sorted(hits, key=lambda p: len(p.parts))[0]

def zip_contains_manifest(zip_path: Path):
    try:
        with zipfile.ZipFile(zip_path, "r") as z:
            return any(Path(n).name == "roi_crop_manifest.csv" for n in z.namelist())
    except Exception:
        return False

def print_input_tree(max_items=200):
    root = Path("/kaggle/input")
    print("Preview of /kaggle/input")
    if not root.exists():
        print("  /kaggle/input does not exist")
        return
    items = list(root.rglob("*"))
    for p in items[:max_items]:
        print(" ", p)
    if len(items) > max_items:
        print(" ...", len(items) - max_items, "additional items")

def resolve_source_path(source_path=None):
    if source_path is not None:
        p = Path(source_path)
        if not p.exists():
            raise FileNotFoundError(f"SOURCE_PATH not found: {p}")
        return p

    input_root = Path("/kaggle/input")
    manifests = list(input_root.rglob("roi_crop_manifest.csv"))
    if manifests:
        return sorted(manifests, key=lambda p: len(p.parts))[0].parent

    zips = sorted(input_root.rglob("*.zip"))
    print("ZIP files found:", len(zips))
    for z in zips:
        print(" -", z)
    candidates = [z for z in zips if zip_contains_manifest(z)]
    if candidates:
        return candidates[0]

    print_input_tree()
    raise FileNotFoundError(
        "No ROI_Crops_256 ZIP or folder containing roi_crop_manifest.csv was found. "
        "Please add ROI_Crops_256_v1_Kaggle.zip or extracted ROI_Crops_256_v1 using Kaggle Add Data."
    )

def prepare_dataset_root(source_path):
    source_path = Path(source_path)
    if source_path.is_dir():
        manifest_path = find_manifest_in_folder(source_path)
        if manifest_path is None:
            raise FileNotFoundError(f"roi_crop_manifest.csv not found in {source_path}")
        return manifest_path.parent

    if source_path.suffix.lower() == ".zip":
        extract_dir = Path("/kaggle/working/ROI_Crops_256_v1_extracted_wave2bis")
        if extract_dir.exists():
            shutil.rmtree(extract_dir)
        extract_dir.mkdir(parents=True, exist_ok=True)
        print("Extracting:", source_path)
        with zipfile.ZipFile(source_path, "r") as z:
            bad = z.testzip()
            if bad is not None:
                raise RuntimeError(f"Corrupted ZIP member: {bad}")
            z.extractall(extract_dir)
        manifest_path = find_manifest_in_folder(extract_dir)
        if manifest_path is None:
            raise FileNotFoundError("roi_crop_manifest.csv not found after extraction")
        return manifest_path.parent

    raise ValueError(f"Unsupported source: {source_path}")

SOURCE = resolve_source_path(SOURCE_PATH)
DATA_ROOT = prepare_dataset_root(SOURCE)
manifest = pd.read_csv(DATA_ROOT / "roi_crop_manifest.csv")

print("SOURCE:", SOURCE)
print("DATA_ROOT:", DATA_ROOT)
print("Manifest:", manifest.shape)
display(manifest.head())

audit = {"source": str(SOURCE), "data_root": str(DATA_ROOT)}
if SOURCE.is_file():
    audit["source_sha256"] = sha256_file(SOURCE)
pd.DataFrame([audit]).to_csv(OUT_DIR / "logs" / "roi256_source_audit.csv", index=False)

In [ ]:
required_cols = ["sample_id", "dataset", "split", "patient_id", "npz_path"]
missing = [c for c in required_cols if c not in manifest.columns]
if missing:
    raise ValueError(f"Missing required manifest columns: {missing}")

print("Counts by dataset/split")
display(manifest.groupby(["dataset", "split"]).size().reset_index(name="n"))

def get_split_df(dataset_keyword, split_name):
    return manifest[(manifest["dataset"].astype(str).str.upper().str.contains(dataset_keyword.upper())) & (manifest["split"].eq(split_name))].copy()

train_df = get_split_df("CBIS", "train")
val_df = get_split_df("CBIS", "validation")
test_df = get_split_df("CBIS", "test")
ext_df = get_split_df("INBREAST", "external_inbreast")

print("Split sizes:", {"train": len(train_df), "validation": len(val_df), "test": len(test_df), "external_inbreast": len(ext_df)})

def patient_set(df):
    return set(df["patient_id"].astype(str).tolist())

splits = {"train": train_df, "validation": val_df, "test": test_df}
for a, b in [("train", "validation"), ("train", "test"), ("validation", "test")]:
    overlap = patient_set(splits[a]) & patient_set(splits[b])
    print(f"Patient overlap {a}/{b}:", len(overlap))
    if overlap:
        raise RuntimeError(f"Patient-level leakage detected between {a} and {b}: {list(sorted(overlap))[:10]}")

# Raw cross-dataset overlap check, mainly for audit traceability.
cbis_patients = patient_set(manifest[manifest["dataset"].astype(str).str.upper().str.contains("CBIS")])
inbreast_patients = patient_set(ext_df)
print("Raw CBIS/INbreast patient_id overlap:", len(cbis_patients & inbreast_patients))

pd.DataFrame([{
    "train_n": len(train_df), "val_n": len(val_df), "test_n": len(test_df), "external_inbreast_n": len(ext_df),
    "cbis_train_val_overlap": len(patient_set(train_df) & patient_set(val_df)),
    "cbis_train_test_overlap": len(patient_set(train_df) & patient_set(test_df)),
    "cbis_val_test_overlap": len(patient_set(val_df) & patient_set(test_df)),
    "cbis_inbreast_raw_overlap": len(cbis_patients & inbreast_patients),
}]).to_csv(OUT_DIR / "logs" / "split_leakage_audit.csv", index=False)

In [ ]:
def resolve_npz_path(row):
    p = Path(str(row["npz_path"]))
    if p.is_absolute() and p.exists():
        return p
    candidates = [DATA_ROOT / p, DATA_ROOT.parent / p]
    for c in candidates:
        if c.exists():
            return c
    raise FileNotFoundError(f"NPZ not found for sample {row.get('sample_id', 'NA')}: {row['npz_path']}")


def load_npz_image_mask(row):
    path = resolve_npz_path(row)
    with np.load(path, allow_pickle=False) as z:
        image = np.squeeze(z["image"]).astype(np.float32)
        mask = np.squeeze(z["mask"]).astype(np.uint8)
    image = np.clip(image, 0, 1).astype(np.float32)
    mask = (mask > 0).astype(np.uint8)
    return image, mask


def apply_clahe_np(image, clip_limit=CLAHE_CLIP_LIMIT, tile_grid_size=CLAHE_TILE_GRID_SIZE):
    if not CV2_AVAILABLE:
        raise RuntimeError("cv2 is required for the optional CLAHE arm.")
    image = np.clip(np.squeeze(image), 0, 1).astype(np.float32)
    u8 = np.round(image * 255.0).astype(np.uint8)
    clahe = cv2.createCLAHE(clipLimit=float(clip_limit), tileGridSize=tuple(tile_grid_size))
    out = clahe.apply(u8).astype(np.float32) / 255.0
    return np.clip(out, 0, 1).astype(np.float32)


def overlay_mask(image, mask, alpha=0.40):
    image = np.clip(image, 0, 1)
    rgb = np.stack([image, image, image], axis=-1).astype(np.float32)
    red = np.zeros_like(rgb)
    red[..., 0] = 1.0
    m = mask.astype(bool)
    rgb[m] = (1 - alpha) * rgb[m] + alpha * red[m]
    return np.clip(rgb, 0, 1)


def build_moderate_aug(image_size=256):
    if not ALBUMENTATIONS_AVAILABLE or not CV2_AVAILABLE:
        return None

    try:
        affine = A.ShiftScaleRotate(
            shift_limit=MOD_SHIFT_LIMIT,
            scale_limit=MOD_SCALE_LIMIT,
            rotate_limit=MOD_ROT_DEG,
            interpolation=cv2.INTER_LINEAR,
            border_mode=cv2.BORDER_CONSTANT,
            value=0,
            mask_value=0,
            p=0.75,
        )
    except TypeError:
        affine = A.ShiftScaleRotate(
            shift_limit=MOD_SHIFT_LIMIT,
            scale_limit=MOD_SCALE_LIMIT,
            rotate_limit=MOD_ROT_DEG,
            interpolation=cv2.INTER_LINEAR,
            border_mode=cv2.BORDER_CONSTANT,
            fill=0,
            fill_mask=0,
            p=0.75,
        )

    return A.Compose([
        A.HorizontalFlip(p=MOD_HFLIP_P),
        A.VerticalFlip(p=MOD_VFLIP_P),
        affine,
        A.RandomBrightnessContrast(
            brightness_limit=MOD_BRIGHTNESS_CONTRAST_LIMIT,
            contrast_limit=MOD_BRIGHTNESS_CONTRAST_LIMIT,
            p=0.35,
        ),
        A.RandomGamma(gamma_limit=MOD_GAMMA_LIMIT, p=0.25),
    ])


def find_local_swin_t_weights():
    if not AUTO_FIND_PRETRAINED_WEIGHTS:
        return None
    root = Path("/kaggle/input")
    if not root.exists():
        return None
    candidates = []
    for pattern in ["*swin_t*.pth", "*swin-t*.pth", "*swin_t*.pt", "*swin-t*.pt"]:
        candidates.extend(root.rglob(pattern))
    candidates = sorted(set(candidates), key=lambda p: (len(str(p)), str(p)))
    if candidates:
        print("Auto-detected local Swin-T weights:", candidates[0])
        return str(candidates[0])
    return None

if PRETRAINED_WEIGHTS_PATH is None:
    PRETRAINED_WEIGHTS_PATH = find_local_swin_t_weights()
print("PRETRAINED_WEIGHTS_PATH:", PRETRAINED_WEIGHTS_PATH)

In [ ]:
def visualize_transfer_preprocessing_examples(df, n=PREPROCESSING_VIS_N, seed=42, save_name="wave2bis_before_after_moderate_aug_examples.png"):
    if not MATPLOTLIB_AVAILABLE:
        print("matplotlib unavailable; skipping visualization.")
        return None
    if len(df) == 0:
        print("No samples available for visualization.")
        return None

    seed_everything(seed)
    df_show = df.sample(min(n, len(df)), random_state=seed).reset_index(drop=True)
    moderate_aug = build_moderate_aug(IMAGE_SIZE)

    n_rows = len(df_show)
    fig, axes = plt.subplots(n_rows, 6, figsize=(18, 3.1 * n_rows))
    if n_rows == 1:
        axes = np.expand_dims(axes, 0)

    for r, (_, row) in enumerate(df_show.iterrows()):
        raw, mask = load_npz_image_mask(row)

        if moderate_aug is not None:
            aug = moderate_aug(image=raw, mask=mask)
            aug_img = np.clip(aug["image"].astype(np.float32), 0, 1)
            aug_mask = (aug["mask"] > 0).astype(np.uint8)
        else:
            image_t = torch.from_numpy(raw).unsqueeze(0).float()
            mask_t = torch.from_numpy(mask.astype(np.float32)).unsqueeze(0).float()
            if random.random() < 0.5:
                image_t = TF.hflip(image_t); mask_t = TF.hflip(mask_t)
            angle = random.uniform(-MOD_ROT_DEG, MOD_ROT_DEG)
            max_t = int(MOD_SHIFT_LIMIT * IMAGE_SIZE)
            translate = (random.randint(-max_t, max_t), random.randint(-max_t, max_t))
            scale = random.uniform(1 - MOD_SCALE_LIMIT, 1 + MOD_SCALE_LIMIT)
            aug_img_t = TF.affine(image_t, angle=angle, translate=translate, scale=scale, shear=[0.0, 0.0], interpolation=InterpolationMode.BILINEAR, fill=0.0)
            aug_mask_t = TF.affine(mask_t, angle=angle, translate=translate, scale=scale, shear=[0.0, 0.0], interpolation=InterpolationMode.NEAREST, fill=0.0)
            aug_img = np.clip(aug_img_t.squeeze(0).numpy().astype(np.float32), 0, 1)
            aug_mask = (aug_mask_t.squeeze(0).numpy() > 0.5).astype(np.uint8)

        imagenet_display = raw.copy()

        panels = [
            (raw, "Raw ROI", "gray"),
            (overlay_mask(raw, mask), "Raw + mask", None),
            (aug_img, "Moderate aug", "gray"),
            (overlay_mask(aug_img, aug_mask), "Aug + mask", None),
            (imagenet_display, "Input before ImageNet norm", "gray"),
            (overlay_mask(imagenet_display, mask), "Input + mask", None),
        ]
        for c, (img, title, cmap) in enumerate(panels):
            ax = axes[r, c]
            if cmap == "gray":
                ax.imshow(img, cmap="gray", vmin=0, vmax=1)
            else:
                ax.imshow(np.clip(img, 0, 1))
            if r == 0:
                ax.set_title(title, fontsize=10)
            ax.axis("off")
        axes[r, 0].set_ylabel(str(row["sample_id"]), fontsize=8)

    fig.suptitle(
        "Wave 2-bis preprocessing control — raw ROI vs moderate train-time augmentation\n"
        "No CLAHE in the main configs; red overlay = ground-truth lesion mask",
        fontsize=14,
    )
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    out = OUT_DIR / "figures" / save_name
    fig.savefig(out, dpi=160)
    plt.close(fig)
    print("Saved:", out)
    display(IPyImage(filename=str(out)))
    return out

if RUN_VISUALIZATION:
    visualize_transfer_preprocessing_examples(train_df, n=PREPROCESSING_VIS_N, seed=42)

In [ ]:
class ROI256TransferDataset(Dataset):
    def __init__(self, df, root, use_clahe=False, augment_mode="none", clip_limit=CLAHE_CLIP_LIMIT, image_size=256):
        self.df = df.reset_index(drop=True).copy()
        self.root = Path(root)
        self.use_clahe = bool(use_clahe)
        self.augment_mode = str(augment_mode)
        self.clip_limit = float(clip_limit)
        self.image_size = int(image_size)
        self.moderate_transform = build_moderate_aug(image_size) if self.augment_mode == "moderate" else None

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image, mask = load_npz_image_mask(row)

        if self.use_clahe:
            image = apply_clahe_np(image, clip_limit=self.clip_limit)

        if self.augment_mode == "moderate":
            if self.moderate_transform is not None:
                augmented = self.moderate_transform(image=image, mask=mask)
                image = np.clip(augmented["image"].astype(np.float32), 0, 1)
                mask = (augmented["mask"] > 0).astype(np.uint8)
            else:
                image_t = torch.from_numpy(image).unsqueeze(0).float()
                mask_t = torch.from_numpy(mask.astype(np.float32)).unsqueeze(0).float()
                image_t, mask_t = self.apply_torch_affine_augmentation(image_t, mask_t, moderate=True)
                image = image_t.squeeze(0).numpy().astype(np.float32)
                mask = (mask_t.squeeze(0).numpy() > 0.5).astype(np.uint8)
        elif self.augment_mode == "light":
            image_t = torch.from_numpy(image).unsqueeze(0).float()
            mask_t = torch.from_numpy(mask.astype(np.float32)).unsqueeze(0).float()
            image_t, mask_t = self.apply_torch_affine_augmentation(image_t, mask_t, moderate=False)
            image = image_t.squeeze(0).numpy().astype(np.float32)
            mask = (mask_t.squeeze(0).numpy() > 0.5).astype(np.uint8)

        image_t = torch.from_numpy(image).unsqueeze(0).float()
        mask_t = torch.from_numpy(mask.astype(np.float32)).unsqueeze(0).float()

        return {
            "image": image_t,
            "mask": mask_t,
            "sample_id": str(row["sample_id"]),
            "patient_id": str(row["patient_id"]),
            "dataset": str(row["dataset"]),
            "split": str(row["split"]),
        }

    def apply_torch_affine_augmentation(self, image_t, mask_t, moderate=False):
        if not TORCHVISION_AVAILABLE:
            return image_t, mask_t
        if random.random() < LIGHT_HFLIP_P:
            image_t = TF.hflip(image_t)
            mask_t = TF.hflip(mask_t)
        if moderate and random.random() < MOD_VFLIP_P:
            image_t = TF.vflip(image_t)
            mask_t = TF.vflip(mask_t)

        rot_deg = MOD_ROT_DEG if moderate else LIGHT_ROT_DEG
        shift_frac = MOD_SHIFT_LIMIT if moderate else LIGHT_TRANSLATE_FRAC
        scale_low = 1 - MOD_SCALE_LIMIT if moderate else LIGHT_SCALE_LOW
        scale_high = 1 + MOD_SCALE_LIMIT if moderate else LIGHT_SCALE_HIGH

        angle = random.uniform(-rot_deg, rot_deg)
        max_t = int(shift_frac * self.image_size)
        translate = (random.randint(-max_t, max_t), random.randint(-max_t, max_t))
        scale = random.uniform(scale_low, scale_high)

        image_t = TF.affine(image_t, angle=angle, translate=translate, scale=scale, shear=[0.0, 0.0], interpolation=InterpolationMode.BILINEAR, fill=0.0)
        mask_t = TF.affine(mask_t, angle=angle, translate=translate, scale=scale, shear=[0.0, 0.0], interpolation=InterpolationMode.NEAREST, fill=0.0)
        mask_t = (mask_t > 0.5).float()
        return image_t, mask_t


def get_wave2bis_config(name):
    configs = {
        "scratch_baseline_rerun": {
            "config": "scratch_baseline_rerun", "use_clahe": False, "augment_mode": "light", "swa": False, "pretrained": False,
            "freeze_pretrained_encoder_epochs": 0,
            "description": "Scratch Swin-Tiny rerun, light augmentation, no CLAHE. Optional repeat of the previous baseline."
        },
        "pretrained_light": {
            "config": "pretrained_light", "use_clahe": False, "augment_mode": "light", "swa": False, "pretrained": True,
            "freeze_pretrained_encoder_epochs": FREEZE_PRETRAINED_ENCODER_EPOCHS,
            "description": "ImageNet-pretrained Swin-Tiny, light augmentation, no CLAHE."
        },
        "pretrained_augmod": {
            "config": "pretrained_augmod", "use_clahe": False, "augment_mode": "moderate", "swa": False, "pretrained": True,
            "freeze_pretrained_encoder_epochs": FREEZE_PRETRAINED_ENCODER_EPOCHS,
            "description": "ImageNet-pretrained Swin-Tiny, moderate augmentation, no CLAHE."
        },
        "pretrained_augmod_swa": {
            "config": "pretrained_augmod_swa", "use_clahe": False, "augment_mode": "moderate", "swa": True, "pretrained": True,
            "freeze_pretrained_encoder_epochs": FREEZE_PRETRAINED_ENCODER_EPOCHS,
            "description": "ImageNet-pretrained Swin-Tiny, moderate augmentation, SWA, no CLAHE."
        },
        "pretrained_augmod_clahe": {
            "config": "pretrained_augmod_clahe", "use_clahe": True, "augment_mode": "moderate", "swa": False, "pretrained": True,
            "freeze_pretrained_encoder_epochs": FREEZE_PRETRAINED_ENCODER_EPOCHS,
            "description": "Optional ablation only: ImageNet-pretrained Swin-Tiny, moderate augmentation, CLAHE."
        },
    }
    if name not in configs:
        raise ValueError(f"Unknown Wave 2-bis config: {name}")
    return configs[name]


def make_loaders(seed, cfg):
    g = torch.Generator()
    g.manual_seed(seed)

    tr = train_df.copy()
    va = val_df.copy()
    te = test_df.copy()
    ex = ext_df.copy()

    if RUN_FAST_DEV:
        tr = tr.sample(min(len(tr), 64), random_state=seed)
        va = va.sample(min(len(va), 32), random_state=seed)
        te = te.sample(min(len(te), 32), random_state=seed)
        if len(ex):
            ex = ex.sample(min(len(ex), 32), random_state=seed)

    common = dict(root=DATA_ROOT, use_clahe=cfg["use_clahe"], clip_limit=CLAHE_CLIP_LIMIT, image_size=IMAGE_SIZE)
    loaders = {
        "train": DataLoader(
            ROI256TransferDataset(tr, augment_mode=cfg["augment_mode"], **common),
            batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True,
            worker_init_fn=worker_init_fn, generator=g
        ),
        "train_eval": DataLoader(
            ROI256TransferDataset(tr, augment_mode="none", **common),
            batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True,
            worker_init_fn=worker_init_fn, generator=g
        ),
        "validation": DataLoader(
            ROI256TransferDataset(va, augment_mode="none", **common),
            batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True,
            worker_init_fn=worker_init_fn, generator=g
        ),
        "test": DataLoader(
            ROI256TransferDataset(te, augment_mode="none", **common),
            batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True,
            worker_init_fn=worker_init_fn, generator=g
        ),
        "external_inbreast": DataLoader(
            ROI256TransferDataset(ex, augment_mode="none", **common),
            batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True,
            worker_init_fn=worker_init_fn, generator=g
        ),
    }
    return loaders

example_cfg = get_wave2bis_config(CONFIGS_TO_RUN[0]) if CONFIGS_TO_RUN else get_wave2bis_config("pretrained_light")
loaders_test = make_loaders(42, example_cfg)
for k, v in loaders_test.items():
    print(k, len(v.dataset), "samples")
b = next(iter(loaders_test["train"]))
print("Sanity-check batch:", b["image"].shape, b["mask"].shape, b["image"].min().item(), b["image"].max().item())

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, groups=8):
        super().__init__()
        groups = min(groups, out_ch)
        while out_ch % groups != 0 and groups > 1:
            groups -= 1
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(groups, out_ch),
            nn.SiLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(groups, out_ch),
            nn.SiLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)

class UpConv(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, 2, 2)
        self.conv = ConvBlock(out_ch + skip_ch, out_ch)
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))


def _extract_state_dict(obj):
    """Accept raw torchvision state_dict or common checkpoint wrappers."""
    if isinstance(obj, dict):
        for key in ["state_dict", "model_state_dict", "model", "weights"]:
            if key in obj and isinstance(obj[key], dict):
                return obj[key]
    return obj


def load_local_swin_t_weights(swin_model, weights_path):
    """Load torchvision Swin-T ImageNet weights from a local Kaggle input file. No Internet is used."""
    if weights_path is None or str(weights_path).strip() == "":
        raise RuntimeError(
            "A pretrained config was requested, but PRETRAINED_WEIGHTS_PATH is not set. "
            "Add the torchvision Swin-T ImageNet .pth file as a Kaggle input, e.g. swin_t-704ceda3.pth, "
            "or disable pretrained configs."
        )
    weights_path = Path(weights_path)
    if not weights_path.exists():
        raise FileNotFoundError(f"PRETRAINED_WEIGHTS_PATH does not exist: {weights_path}")

    state = torch.load(weights_path, map_location="cpu")
    state = _extract_state_dict(state)
    if not isinstance(state, dict):
        raise RuntimeError("Could not extract a state_dict from the pretrained weights file.")

    cleaned = {}
    for k, v in state.items():
        nk = k
        for prefix in ["module.", "swin.", "encoder."]:
            if nk.startswith(prefix):
                nk = nk[len(prefix):]
        cleaned[nk] = v

    try:
        swin_model.load_state_dict(cleaned, strict=True)
        print(f"Loaded local Swin-T weights strictly from: {weights_path}")
        return
    except RuntimeError as e:
        print("Strict load did not fully match. Trying feature-safe non-strict load.")
        print(str(e).split("\n")[0])

    missing, unexpected = swin_model.load_state_dict(cleaned, strict=False)
    blocking_missing = [k for k in missing if k.startswith("features.")]
    blocking_unexpected = [k for k in unexpected if k.startswith("features.")]
    if blocking_missing or blocking_unexpected:
        raise RuntimeError(
            "Local Swin-T weights are incompatible with torchvision.swin_t features. "
            f"blocking_missing={blocking_missing[:10]}, blocking_unexpected={blocking_unexpected[:10]}"
        )
    print(f"Loaded local Swin-T feature weights from: {weights_path}")
    if missing:
        print("Non-blocking missing keys, usually classifier/norm head:", missing[:10])
    if unexpected:
        print("Non-blocking unexpected keys:", unexpected[:10])


class SwinTinyUNet(nn.Module):
    def __init__(self, out_ch=1, pretrained=False):
        super().__init__()
        if not TORCHVISION_AVAILABLE:
            raise RuntimeError("torchvision is required for SwinTinyUNet")
        from torchvision.models import swin_t
        self.pretrained = bool(pretrained)
        self.in_adapter = nn.Conv2d(1, 3, 1, bias=False)
        with torch.no_grad():
            self.in_adapter.weight.fill_(1.0)
        self.register_buffer("mean", torch.tensor([0.485,0.456,0.406]).view(1,3,1,1))
        self.register_buffer("std", torch.tensor([0.229,0.224,0.225]).view(1,3,1,1))
        # Keep weights=None to prevent automatic Internet downloads.
        self.swin = swin_t(weights=None)
        if self.pretrained:
            load_local_swin_t_weights(self.swin, PRETRAINED_WEIGHTS_PATH)
        self.features = self.swin.features
        self.center = ConvBlock(768, 512)
        self.dec3 = UpConv(512, 384, 256)
        self.dec2 = UpConv(256, 192, 128)
        self.dec1 = UpConv(128, 96, 64)
        self.up0a = nn.ConvTranspose2d(64, 32, 2, 2)
        self.c0a = ConvBlock(32, 32)
        self.up0b = nn.ConvTranspose2d(32, 16, 2, 2)
        self.c0b = ConvBlock(16, 16)
        self.out = nn.Conv2d(16, out_ch, 1)

    def _nchw(self, x):
        if x.ndim == 4 and x.shape[1] not in [96, 192, 384, 768]:
            return x.permute(0, 3, 1, 2).contiguous()
        return x

    def forward(self, x):
        y = self.in_adapter(x)
        y = (y - self.mean) / self.std
        feats = []
        for i, layer in enumerate(self.features):
            y = layer(y)
            if i in [1, 3, 5, 7]:
                feats.append(self._nchw(y))
        if len(feats) != 4:
            raise RuntimeError(f"Unexpected Swin feature count: {len(feats)}")
        s1, s2, s3, s4 = feats
        x = self.center(s4)
        x = self.dec3(x, s3)
        x = self.dec2(x, s2)
        x = self.dec1(x, s1)
        x = self.c0a(self.up0a(x))
        x = self.c0b(self.up0b(x))
        if x.shape[-2:] != (IMAGE_SIZE, IMAGE_SIZE):
            x = F.interpolate(x, size=(IMAGE_SIZE, IMAGE_SIZE), mode="bilinear", align_corners=False)
        return self.out(x)


def build_model(cfg):
    return SwinTinyUNet(pretrained=cfg["pretrained"])


def make_optimizer(model, cfg):
    enc, dec = [], []
    for n, p in model.named_parameters():
        if n.startswith("features") or n.startswith("swin"):
            enc.append(p)
        else:
            dec.append(p)
    encoder_lr = PRETRAINED_ENCODER_LR if cfg.get("pretrained", False) else ENCODER_LR
    return torch.optim.AdamW(
        [{"params": enc, "lr": encoder_lr}, {"params": dec, "lr": BASE_LR}],
        weight_decay=WEIGHT_DECAY,
    )


def set_encoder_trainable(model, trainable: bool):
    for n, p in model.named_parameters():
        if n.startswith("features") or n.startswith("swin"):
            p.requires_grad = bool(trainable)


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def has_batchnorm(model):
    return any(isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d, nn.SyncBatchNorm)) for m in model.modules())

# Skip this check only for visualization-only runs or when local pretrained weights are unavailable.
if RUN_TRAINING:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if any(get_wave2bis_config(c)["pretrained"] for c in CONFIGS_TO_RUN) and PRETRAINED_WEIGHTS_PATH is None:
        raise RuntimeError(
            "Pretrained configs are enabled but no local Swin-T weights were found. "
            "Add swin_t-704ceda3.pth as a Kaggle input or remove pretrained configs from CONFIGS_TO_RUN."
        )
    for cfg_name in sorted(set(CONFIGS_TO_RUN)):
        cfg = get_wave2bis_config(cfg_name)
        try:
            m = build_model(cfg).to(device)
            with torch.no_grad():
                y = m(torch.randn(1, 1, IMAGE_SIZE, IMAGE_SIZE, device=device))
            print(cfg_name, "params", count_params(m), "out", tuple(y.shape), "has_batchnorm", has_batchnorm(m))
            del m
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception as e:
            print("[MODEL ERROR]", cfg_name, repr(e))
            raise
else:
    print("RUN_TRAINING=False: skipping model construction sanity check.")

In [ ]:
class CombinedLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super().__init__()
        self.smooth = smooth
    def forward(self, logits, target):
        bce = F.binary_cross_entropy_with_logits(logits, target)
        prob = torch.sigmoid(logits)
        dims = (1,2,3)
        inter = (prob * target).sum(dims)
        dice = (2*inter + self.smooth) / (prob.sum(dims) + target.sum(dims) + self.smooth)
        dice_loss = 1 - dice.mean()
        tp = (prob * target).sum(dims)
        fp = (prob * (1-target)).sum(dims)
        fn = ((1-prob) * target).sum(dims)
        tversky = (tp + self.smooth) / (tp + 0.3*fp + 0.7*fn + self.smooth)
        focal_tversky = torch.pow(1 - tversky, 0.75).mean()
        return 0.5*bce + 0.3*dice_loss + 0.2*focal_tversky

def binary_metrics(pred, target, eps=1e-7):
    pred = pred.astype(bool)
    target = target.astype(bool)
    tp = np.logical_and(pred, target).sum()
    fp = np.logical_and(pred, ~target).sum()
    fn = np.logical_and(~pred, target).sum()
    p = pred.sum()
    t = target.sum()
    dice = (2*tp + eps) / (p + t + eps) if t > 0 else (1.0 if p == 0 else 0.0)
    iou = (tp + eps) / (tp + fp + fn + eps) if t > 0 else (1.0 if p == 0 else 0.0)
    precision = (tp + eps) / (tp + fp + eps)
    recall = (tp + eps) / (tp + fn + eps) if t > 0 else np.nan
    tn = np.logical_and(~pred, ~target).sum()
    return {
        "dice": float(dice), "iou": float(iou), "precision": float(precision), "recall": float(recall),
        "empty_pred": int(p == 0), "empty_target": int(t == 0),
        "pred_area": int(p), "target_area": int(t),
        "tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn)
    }

def hd95_asd(pred, target):
    if not SCIPY_AVAILABLE:
        return {"hd95": np.nan, "asd": np.nan}
    pred = pred.astype(bool)
    target = target.astype(bool)
    if pred.sum() == 0 or target.sum() == 0:
        return {"hd95": np.nan, "asd": np.nan}
    pb = pred ^ binary_erosion(pred)
    tb = target ^ binary_erosion(target)
    if pb.sum() == 0 or tb.sum() == 0:
        return {"hd95": np.nan, "asd": np.nan}
    dt_t = distance_transform_edt(~tb)
    dt_p = distance_transform_edt(~pb)
    d = np.concatenate([dt_t[pb], dt_p[tb]]).astype(np.float32)
    return {"hd95": float(np.percentile(d, 95)), "asd": float(d.mean())}

In [ ]:
@torch.no_grad()
def collect_probs(model, loader, device):
    model.eval()
    rows, probs, masks = [], [], []
    for batch in loader:
        x = batch["image"].to(device, non_blocking=True)
        logits = model(x)
        pr = torch.sigmoid(logits).cpu().numpy()
        ma = batch["mask"].cpu().numpy()
        for i in range(x.shape[0]):
            rows.append({
                "sample_id": batch["sample_id"][i],
                "patient_id": batch["patient_id"][i],
                "dataset": batch["dataset"][i],
                "split": batch["split"][i],
            })
            probs.append(pr[i,0].astype(np.float32))
            masks.append(ma[i,0].astype(np.uint8))
    return rows, probs, masks


def evaluate_probs(rows, probs, masks, threshold, config_name, seed, split_eval, policy):
    out = []
    for row, prob, mask in zip(rows, probs, masks):
        pred = (prob >= threshold).astype(np.uint8)
        met = binary_metrics(pred, mask)
        surf = hd95_asd(pred, mask)
        out.append({
            **row, "model": "swin_tiny_unet", "config": config_name, "seed": seed, "split_eval": split_eval,
            "threshold": float(threshold), "threshold_policy": policy,
            **met, **surf
        })
    return pd.DataFrame(out)


def aggregate(df):
    result = {"n": len(df)}
    for c in ["dice", "iou", "precision", "recall", "hd95", "asd", "empty_pred"]:
        result[c] = float(np.nanmean(df[c])) if c in df and len(df) else np.nan
    return result


def select_threshold(config_name, seed, val_rows, val_probs, val_masks):
    rows = []
    for th in THRESHOLDS:
        df = evaluate_probs(val_rows, val_probs, val_masks, th, config_name, seed, "validation", "candidate")
        rows.append({"threshold": th, **aggregate(df)})
    sweep = pd.DataFrame(rows)
    sweep["dist_to_05"] = (sweep["threshold"] - 0.5).abs()
    sweep = sweep.sort_values(["dice", "iou", "dist_to_05", "threshold"], ascending=[False, False, True, True])
    best = float(sweep.iloc[0]["threshold"])
    return best, sweep.sort_values("threshold")


def save_examples(config_name, seed, split_name, rows, probs, masks, threshold, n=16):
    if not MATPLOTLIB_AVAILABLE or len(rows) == 0:
        return None
    cfg = get_wave2bis_config(config_name)
    idxs = np.linspace(0, len(rows)-1, min(n, len(rows))).astype(int)
    n_rows = len(idxs)
    plt.figure(figsize=(12, max(3.0, n_rows * 3.0)))
    for r, idx in enumerate(idxs):
        sid = rows[idx]["sample_id"]
        match = manifest.loc[manifest["sample_id"].astype(str).eq(str(sid))]
        if len(match):
            raw, _ = load_npz_image_mask(match.iloc[0])
            img = apply_clahe_np(raw, CLAHE_CLIP_LIMIT) if cfg["use_clahe"] else raw
        else:
            img = masks[idx].astype(np.float32)
        pred = (probs[idx] >= threshold).astype(np.uint8)
        mask = masks[idx].astype(np.uint8)

        overlay = np.stack([img, img, img], axis=-1)
        overlay[..., 0] = np.maximum(overlay[..., 0], pred * 1.0)
        overlay[..., 1] = np.maximum(overlay[..., 1], mask * 0.95)

        ax = plt.subplot(n_rows, 3, r*3 + 1)
        ax.imshow(img, cmap="gray", vmin=0, vmax=1)
        ax.set_title("Input ROI", fontsize=9)
        ax.axis("off")

        ax = plt.subplot(n_rows, 3, r*3 + 2)
        ax.imshow(mask, cmap="gray", vmin=0, vmax=1)
        ax.set_title("Ground truth mask", fontsize=9)
        ax.axis("off")

        ax = plt.subplot(n_rows, 3, r*3 + 3)
        ax.imshow(np.clip(overlay, 0, 1))
        ax.set_title("Prediction overlay", fontsize=9)
        ax.axis("off")

    plt.suptitle(
        f"Qualitative segmentation — {config_name} | seed {seed} | {split_name} | threshold={threshold:.3f}\n"
        "Green: ground truth, Red: prediction, Yellow: overlap",
        fontsize=13
    )
    plt.tight_layout(rect=[0, 0, 1, 0.98])
    out = OUT_DIR / "figures" / f"{config_name}_seed{seed}_{split_name}_qualitative_segmentation.png"
    plt.savefig(out, dpi=150)
    plt.close()
    return out

In [ ]:
def lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return float(epoch + 1) / max(1, WARMUP_EPOCHS)
    progress = (epoch - WARMUP_EPOCHS) / max(1, EPOCHS - WARMUP_EPOCHS)
    return 0.5 * (1 + math.cos(math.pi * progress))


def train_one_epoch(model, loader, optimizer, criterion, scaler, device):
    model.train()
    total, n = 0.0, 0
    optimizer.zero_grad(set_to_none=True)
    for batch in loader:
        x = batch["image"].to(device, non_blocking=True)
        y = batch["mask"].to(device, non_blocking=True)
        with autocast(enabled=AMP and device.type == "cuda"):
            logits = model(x)
            loss = criterion(logits, y)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        total += float(loss.detach().cpu()) * x.size(0)
        n += x.size(0)
    return total / max(n, 1)


@torch.no_grad()
def eval_loss(model, loader, criterion, device):
    model.eval()
    total, n = 0.0, 0
    for batch in loader:
        x = batch["image"].to(device, non_blocking=True)
        y = batch["mask"].to(device, non_blocking=True)
        with autocast(enabled=AMP and device.type == "cuda"):
            loss = criterion(model(x), y)
        total += float(loss.detach().cpu()) * x.size(0)
        n += x.size(0)
    return total / max(n, 1)


def run_wave2bis_experiment(config_name, seed):
    cfg = get_wave2bis_config(config_name)
    print(f"\n===== START {config_name} seed={seed} | {now()} =====")
    print(cfg["description"])
    seed_everything(seed)
    loaders = make_loaders(seed, cfg)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = build_model(cfg).to(device)

    freeze_epochs = int(cfg.get("freeze_pretrained_encoder_epochs", 0)) if cfg.get("pretrained", False) else 0
    if freeze_epochs > 0:
        print(f"Freezing pretrained encoder for {freeze_epochs} epochs.")
        set_encoder_trainable(model, False)

    optimizer = make_optimizer(model, cfg)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
    criterion = CombinedLoss()
    scaler = GradScaler(enabled=AMP and device.type == "cuda")

    ckpt_path = OUT_DIR / "checkpoints" / f"{config_name}_seed{seed}_best.pt"
    final_ckpt_path = OUT_DIR / "checkpoints" / f"{config_name}_seed{seed}_final_eval.pt"

    best_val = float("inf")
    best_epoch = -1
    bad = 0
    history = []

    use_swa = bool(cfg["swa"])
    swa_start = max(1, int(EPOCHS * SWA_START_FRAC))
    swa_model = AveragedModel(model) if use_swa else None
    swa_scheduler = SWALR(optimizer, swa_lr=SWA_LR) if use_swa else None
    min_epochs_before_stop = swa_start + 5 if use_swa else 1

    for epoch in range(1, EPOCHS + 1):
        if freeze_epochs > 0 and epoch == freeze_epochs + 1:
            print("Unfreezing pretrained encoder.")
            set_encoder_trainable(model, True)

        t0 = time.time()
        tr = train_one_epoch(model, loaders["train"], optimizer, criterion, scaler, device)
        va = eval_loss(model, loaders["validation"], criterion, device)

        if use_swa and epoch >= swa_start:
            swa_model.update_parameters(model)
            swa_scheduler.step()
            lr_current = optimizer.param_groups[0]["lr"]
            swa_active = True
        else:
            scheduler.step()
            lr_current = optimizer.param_groups[0]["lr"]
            swa_active = False

        history.append({
            "config": config_name, "seed": seed, "epoch": epoch,
            "train_loss": tr, "val_loss": va, "lr_encoder": optimizer.param_groups[0]["lr"],
            "lr_decoder": optimizer.param_groups[1]["lr"], "swa_active": swa_active,
            "encoder_frozen": freeze_epochs > 0 and epoch <= freeze_epochs,
            "seconds": time.time()-t0
        })
        print(f"epoch {epoch:03d} train={tr:.5f} val={va:.5f} lr_enc={optimizer.param_groups[0]['lr']:.2e} lr_dec={optimizer.param_groups[1]['lr']:.2e} swa={swa_active}")

        if va < best_val - 1e-5:
            best_val = va
            best_epoch = epoch
            bad = 0
            torch.save({
                "model": "swin_tiny_unet", "config": config_name, "seed": seed, "epoch": epoch,
                "state_dict": model.state_dict(), "cfg": cfg, "best_val_loss": best_val,
                "pretrained_weights_path": PRETRAINED_WEIGHTS_PATH,
            }, ckpt_path)
        else:
            bad += 1

        pd.DataFrame(history).to_csv(OUT_DIR / "metrics" / f"{config_name}_seed{seed}_history.csv", index=False)
        if bad >= PATIENCE and epoch >= min_epochs_before_stop:
            print("Early stopping. Best epoch:", best_epoch, "best validation loss:", best_val)
            break

    if use_swa:
        eval_model = swa_model.to(device)
        if has_batchnorm(eval_model):
            print("BatchNorm detected. Updating BN statistics with non-augmented train loader.")
            update_bn(loaders["train_eval"], eval_model, device=device)
        eval_state = eval_model.module.state_dict() if hasattr(eval_model, "module") else eval_model.state_dict()
        eval_kind = "swa_averaged"
        torch.save({
            "model": "swin_tiny_unet", "config": config_name, "seed": seed,
            "state_dict": eval_state, "cfg": cfg, "eval_kind": eval_kind,
            "best_epoch_before_swa": best_epoch, "best_val_loss_before_swa": best_val,
            "pretrained_weights_path": PRETRAINED_WEIGHTS_PATH,
        }, final_ckpt_path)
    else:
        ckpt = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(ckpt["state_dict"], strict=True)
        eval_model = model
        eval_kind = "best_val_loss"
        shutil.copy2(ckpt_path, final_ckpt_path)

    caches = {}
    for split in ["train_eval", "validation", "test", "external_inbreast"]:
        if len(loaders[split].dataset) == 0:
            caches[split] = ([], [], [])
        else:
            caches[split] = collect_probs(eval_model, loaders[split], device)
            rows, probs, masks = caches[split]
            np.save(OUT_DIR / "probabilities" / f"{config_name}_seed{seed}_{split}_probs.npy", np.stack(probs).astype(np.float32))

    val_rows, val_probs, val_masks = caches["validation"]
    selected_th, sweep = select_threshold(config_name, seed, val_rows, val_probs, val_masks)
    sweep["config"] = config_name
    sweep["seed"] = seed
    sweep["eval_kind"] = eval_kind
    sweep.to_csv(OUT_DIR / "metrics" / f"{config_name}_seed{seed}_threshold_sweep_validation.csv", index=False)

    detailed_parts = []
    for split in ["train_eval", "validation", "test", "external_inbreast"]:
        rows, probs, masks = caches[split]
        if len(rows) == 0:
            continue
        split_label = "train" if split == "train_eval" else split
        for th, policy in [(selected_th, "selected_on_cbis_validation"), (0.50, "fixed_0_50")]:
            detailed_parts.append(evaluate_probs(rows, probs, masks, th, config_name, seed, split_label, policy))
        fig_path = save_examples(config_name, seed, split_label, rows, probs, masks, selected_th, n=SAVE_QUALITATIVE_N)
        if fig_path is not None:
            print("Saved qualitative segmentation panel:", fig_path)

    detailed = pd.concat(detailed_parts, ignore_index=True)
    detailed["eval_kind"] = eval_kind
    detailed.to_csv(OUT_DIR / "metrics" / f"{config_name}_seed{seed}_detailed_metrics.csv", index=False)

    summaries = []
    for (split, policy), g in detailed.groupby(["split_eval", "threshold_policy"]):
        summaries.append({
            "model": "swin_tiny_unet", "config": config_name, "seed": seed, "split": split,
            "threshold_policy": policy,
            "selected_threshold": selected_th if policy == "selected_on_cbis_validation" else 0.50,
            "eval_kind": eval_kind,
            "best_epoch": best_epoch, "best_val_loss": best_val,
            "sha256_final_checkpoint": sha256_file(final_ckpt_path),
            **aggregate(g)
        })
    summary = pd.DataFrame(summaries)
    summary.to_csv(OUT_DIR / "metrics" / f"{config_name}_seed{seed}_summary.csv", index=False)
    print(f"===== END {config_name} seed={seed} | selected threshold={selected_th} | eval={eval_kind} =====")

    del model, eval_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return summary, sweep

In [ ]:
all_summaries = []
all_sweeps = []

if RUN_TRAINING:
    for config_name in CONFIGS_TO_RUN:
        for seed in TRAINING_SEEDS:
            summary, sweep = run_wave2bis_experiment(config_name, seed)
            all_summaries.append(summary)
            all_sweeps.append(sweep)

    if all_summaries:
        pd.concat(all_summaries, ignore_index=True).to_csv(OUT_DIR / "metrics" / "all_wave2bis_seed_summaries.csv", index=False)
    if all_sweeps:
        pd.concat(all_sweeps, ignore_index=True).to_csv(OUT_DIR / "metrics" / "all_wave2bis_threshold_sweeps.csv", index=False)
else:
    print("RUN_TRAINING=False: only preprocessing visualization and dataset QC were executed.")

In [ ]:
def mean_std_str(x):
    x = pd.to_numeric(x, errors="coerce")
    return f"{np.nanmean(x):.3f} ± {np.nanstd(x, ddof=1):.3f}" if x.notna().sum() > 1 else f"{np.nanmean(x):.3f}"

summary_path = OUT_DIR / "metrics" / "all_wave2bis_seed_summaries.csv"
if summary_path.exists():
    summaries = pd.read_csv(summary_path)
else:
    summary_files = sorted((OUT_DIR / "metrics").glob("*_summary.csv"))
    summaries = pd.concat([pd.read_csv(p) for p in summary_files], ignore_index=True) if summary_files else pd.DataFrame()

if len(summaries):
    selected = summaries[summaries["threshold_policy"].eq("selected_on_cbis_validation")].copy()
    group_cols = ["config", "split"]
    rows = []
    for (cfg, split), g in selected.groupby(group_cols):
        row = {"config": cfg, "split": split, "n_seeds": g["seed"].nunique()}
        for metric in ["selected_threshold", "dice", "iou", "precision", "recall", "hd95", "asd", "empty_pred"]:
            row[metric + "_mean"] = float(np.nanmean(g[metric]))
            row[metric + "_std"] = float(np.nanstd(g[metric], ddof=1)) if g[metric].notna().sum() > 1 else 0.0
            row[metric + "_mean_std"] = mean_std_str(g[metric])
        rows.append(row)
    agg = pd.DataFrame(rows)
    agg.to_csv(OUT_DIR / "wave2bis_results.csv", index=False)

    wide = agg.pivot(index="config", columns="split", values=["dice_mean", "hd95_mean", "asd_mean", "selected_threshold_mean"])
    wide.columns = [f"{a}_{b}" for a, b in wide.columns]
    wide = wide.reset_index()
    wide["delta_cbis_test_dice_vs_baseline"] = wide.get("dice_mean_test", np.nan) - BASELINE_REFERENCE["cbis_test_dice"]
    wide["delta_inbreast_dice_vs_baseline"] = wide.get("dice_mean_external_inbreast", np.nan) - BASELINE_REFERENCE["inbreast_dice"]
    wide["delta_inbreast_hd95_vs_baseline"] = BASELINE_REFERENCE["inbreast_hd95"] - wide.get("hd95_mean_external_inbreast", np.nan)

    if "dice_mean_validation" in wide.columns:
        wide = wide.sort_values("dice_mean_validation", ascending=False).reset_index(drop=True)
    wide.to_csv(OUT_DIR / "wave2bis_results_wide.csv", index=False)

    payload = {
        "created_at": now(),
        "baseline_reference": BASELINE_REFERENCE,
        "methodological_guardrail": "Thresholds selected only on CBIS-DDSM validation, then frozen for CBIS-DDSM test and INbreast external.",
        "results_long": agg.to_dict(orient="records"),
        "results_wide": wide.to_dict(orient="records"),
    }
    with open(OUT_DIR / "wave2bis_results.json", "w") as f:
        json.dump(payload, f, indent=2)

    md = []
    md.append("# Wave 2-bis results — Transfer learning isolated\n")
    md.append("## Protocol guardrail\n")
    md.append("All thresholds were selected only on CBIS-DDSM validation and then frozen for CBIS-DDSM test and INbreast external validation. No test/external metric was used for threshold or configuration selection.\n")
    md.append("## Baseline reference\n")
    md.append(f"Previous Swin-Tiny-U-Net scratch baseline: CBIS test Dice={BASELINE_REFERENCE['cbis_test_dice']:.3f}, INbreast Dice={BASELINE_REFERENCE['inbreast_dice']:.3f}, INbreast HD95={BASELINE_REFERENCE['inbreast_hd95']:.2f}.\n")
    md.append("## Aggregated results\n")
    md.append(wide.to_markdown(index=False))
    md.append("\n\n## Recommendation rule\n")
    md.append("The final configuration should be chosen from validation behavior first, then accepted only if it does not materially degrade sealed CBIS test performance and improves or preserves INbreast external performance relative to the baseline.\n")
    with open(OUT_DIR / "wave2bis_results.md", "w") as f:
        f.write("\n".join(md))

    print("Saved:", OUT_DIR / "wave2bis_results.csv")
    print("Saved:", OUT_DIR / "wave2bis_results_wide.csv")
    print("Saved:", OUT_DIR / "wave2bis_results.json")
    print("Saved:", OUT_DIR / "wave2bis_results.md")
    display(wide)
else:
    print("No summaries found yet. Run training first or set RUN_TRAINING=True.")

In [ ]:
if MATPLOTLIB_AVAILABLE and (OUT_DIR / "wave2bis_results_wide.csv").exists():
    wide = pd.read_csv(OUT_DIR / "wave2bis_results_wide.csv")
    if "dice_mean_external_inbreast" in wide.columns:
        plot_df = wide.sort_values("dice_mean_external_inbreast", ascending=False)
        plt.figure(figsize=(10, max(4, len(plot_df) * 0.6)))
        plt.barh(plot_df["config"], plot_df["dice_mean_external_inbreast"])
        plt.axvline(BASELINE_REFERENCE["inbreast_dice"], linestyle="--", linewidth=1.5, label="Previous scratch baseline INbreast Dice")
        plt.xlabel("INbreast Dice")
        plt.title("Wave 2-bis configurations ranked by external INbreast Dice")
        plt.legend()
        plt.tight_layout()
        out = OUT_DIR / "figures" / "wave2bis_inbreast_dice_ranking.png"
        plt.savefig(out, dpi=160)
        plt.close()
        display(IPyImage(filename=str(out)))

zip_path = Path("/kaggle/working/ROI256_WAVE2BIS_TRANSFER_RESULTS.zip")
if OUT_DIR.exists():
    if zip_path.exists():
        zip_path.unlink()
    shutil.make_archive(str(zip_path).replace(".zip", ""), "zip", OUT_DIR)
    print("Final ZIP:", zip_path)
else:
    print("OUT_DIR does not exist; nothing to zip.")

## Required inputs

The workflow requires the prepared mammography ROI dataset and a local torchvision Swin-T ImageNet checkpoint. The checkpoint path can be specified in the configuration cell when automatic discovery is unavailable.
